# Notebook 02 — Phase 1: AlexNet Evaluation

Evaluate the trained AlexNet on the held-out test set.  
Produces: accuracy, precision, recall, F1, confusion matrix, ROC curve.

**Prerequisites**: Run notebooks 00 and 01 first.

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)
import config

## 1. Load Best Checkpoint & Run Evaluation

In [ ]:
from phase1_alexnet.evaluate import run_evaluation

results = run_evaluation(
    checkpoint_path=config.ALEXNET_CHECKPOINT,
    processed_dir=config.PROCESSED_DATA_DIR,
    plots_dir=config.PLOTS_DIR,
)

## 2. Metrics Summary

In [ ]:
print('=== Test Set Metrics ===')
print(f'Accuracy : {results["accuracy"]:.4f}')
print(f'Precision: {results["precision"]:.4f}')
print(f'Recall   : {results["recall"]:.4f}')
print(f'F1-Score : {results["f1"]:.4f}')
if results.get('roc_auc'):
    print(f'ROC-AUC  : {results["roc_auc"]:.4f}')

# Target checks
targets = {'Accuracy': 0.90, 'Precision': 0.88, 'Recall': 0.88, 'F1': 0.87}
metrics = {'Accuracy': results['accuracy'], 'Precision': results['precision'],
           'Recall': results['recall'], 'F1': results['f1']}
print('\n=== Target Assessment ===')
for name, target in targets.items():
    val = metrics[name]
    status = 'PASS' if val >= target else 'BELOW TARGET'
    print(f'  {name}: {val:.4f} (target >= {target}) — {status}')

## 3. Confusion Matrix

In [ ]:
from utils.plot_utils import plot_confusion_matrix

plot_confusion_matrix(
    results['confusion_matrix'],
    config.CLASS_NAMES,
    title='AlexNet — Test Set Confusion Matrix'
)

## 4. ROC Curve

In [ ]:
from utils.plot_utils import plot_roc_curve

auc = plot_roc_curve(results['all_probs'], results['all_labels'])
print(f'AUC: {auc:.4f}')

## 5. Visualise Predictions — Correct & Incorrect

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from phase1_alexnet.dataset import LandslideDataset, get_test_transforms

test_ds = LandslideDataset(config.PROCESSED_DATA_DIR, 'test', None)
preds = results['all_preds']
labels = results['all_labels']
probs = results['all_probs']

correct_idx  = np.where(preds == labels)[0][:4]
incorrect_idx = np.where(preds != labels)[0][:4]

def show_predictions(indices, title):
    if len(indices) == 0:
        print(f'No examples for: {title}')
        return
    fig, axes = plt.subplots(1, len(indices), figsize=(4*len(indices), 4))
    if len(indices) == 1: axes = [axes]
    for ax, i in zip(axes, indices):
        img_path = test_ds.samples[i][0]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        true_lbl = config.CLASS_NAMES[labels[i]]
        pred_lbl = config.CLASS_NAMES[preds[i]]
        conf = probs[i] if preds[i] == 1 else 1 - probs[i]
        ax.set_title(f'True: {true_lbl}\nPred: {pred_lbl} ({conf:.2f})', fontsize=9)
        ax.axis('off')
    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_predictions(correct_idx,  'Correct Predictions')
show_predictions(incorrect_idx, 'Incorrect Predictions')